##Bronze to Silver: Fact data cleansing and transformation

In [0]:
from pyspark.sql.types import StringType, IntegerType, DateType, BooleanType
import pyspark.sql.functions as F

In [0]:
catalog_name = 'ecommerce'

df_bronze_orders = spark.table(f'{catalog_name}.bronze.brz_order_items')
display(df_bronze_orders)

In [0]:
#removing duplicates
df_silver_orders = df_bronze_orders.dropDuplicates(['order_id', 'item_seq'])

#standardizing quantity, unit_price, discount_percentage column
df_silver_orders = df_silver_orders.withColumn('quantity', F.when(F.col('quantity') == 'Two', 2).otherwise(F.col('quantity')).cast('int'))
df_silver_orders = df_silver_orders.withColumn('unit_price', F.regexp_replace(F.col('unit_price'), '[$]', '').cast('double'))
df_silver_orders = df_silver_orders.withColumn('discount_pct', F.regexp_replace(F.col('discount_pct'), '%', '').cast('double'))

#coupon code standardization
df_silver_orders = df_silver_orders.withColumn('coupon_code', F.lower(F.trim(F.col('coupon_code'))))

#channel processing
df_silver_orders = df_silver_orders.withColumn('channel', F.when(F.col('channel') == 'app', 'Mobile').when(F.col('channel') == 'web', 'Website').otherwise(F.col('channel')))

#tax_amount standardization
df_silver_orders = df_silver_orders.withColumn("tax_amount", F.regexp_replace("tax_amount", r"[^0-9.\-]", "").cast("double"))

#Transformation : Add processed time 
df_silver_orders = df_silver_orders.withColumn("processed_time", F.current_timestamp())
display(df_silver_orders)

In [0]:
df_silver_orders.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(f"{catalog_name}.silver.slv_order_items")